# Dephased-IC dataset generation

Generates the **normal** and **seizure** datasets on `single-knob-dephased-ic`.

Each recording warm-starts from a saved network state instead of
`h.finitialize(-65)`, which removes the initialization burst that fires at ~4.9 s
in every flagship recording. Everything else matches the flagship.

Edit **CONFIG** below, then run the cells in order. Output goes to
`notebooks/NEURON data parallel/dephased_ic/{normal,seizure}/`.


In [ ]:
import os, sys
REPO_ROOT = os.path.abspath('..')
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'analysis'), os.path.join(REPO_ROOT, 'inference')):
    if p not in sys.path:
        sys.path.insert(0, p)

import dephase_nb as nb          # all the plumbing lives here
from neuron_simulation.topology import NeuronWeightParameters


In [ ]:
# ==========================================================================
# CONFIG - the only cell you need to edit.
# ONE fixed network. Normal vs seizure differ by a SINGLE parameter:
#   sahp_ainc_slow (slow-AHP / M-current strength).
# ==========================================================================
SAHP_NORMAL, SAHP_SEIZURE = 0.01, 0.004   # the ONLY thing that differs between states

wp = NeuronWeightParameters()
wp.within_exc_range  = (0.0010, 0.0022)   # exc weight, same cluster
wp.between_exc_range = (0.0008, 0.0016)   # exc weight, across clusters
wp.within_inh_range  = (0.0025, 0.0055)   # inh weight, same cluster
wp.between_inh_range = (0.0020, 0.0040)   # inh weight, across clusters
wp.use_lognormal     = True
wp.lognormal_sigma   = 0.5

CONFIG = {

    # ---- which network -----------------------------------------------------
    # True  : reuse the flagship's already-built graph, so this dataset differs
    #         from the 200-recording dataset ONLY in the initial condition.
    # False : build a fresh graph from CONFIG['topology'] below. The graph is
    #         cached, because the warm-start library and the recordings must
    #         share one graph -- change this and the libraries need rebuilding.
    'use_flagship_topology': True,
    'topology_kind': 'lognormal',
    'topology_tag': 'custom',             # names the cached graph file

    'topology': dict(                     # only used when the flag above is False
        num_clusters=50, neurons_per_cluster_range=(4, 40),
        inhibitory_probability=0.2, cluster_radius=1.0, space_size=15.0, seed=1,
        decay_sigma=3.0, max_connection_distance=6.0,
        cell_type_specific=True,
        p_ee_within=0.2, p_ee_between=0.1, p_ei_within=0.20, p_ei_between=0.08,
        p_ie_within=0.40, p_ii_within=0.50,
        within_cluster_prob=0.25, between_cluster_prob=0.06,
        ln_sigma=0.5, target_density=None, weight_params=wp,
    ),

    # ---- cell + synapse parameters -----------------------------------------
    # These are the flagship's ACTUAL values. Preflight reports any you change,
    # and warns that the dataset is then no longer an IC-only comparison.
    'build': dict(
        synapse_model='ampa_nmda', exc_tau=5.0, tau_nmda=350.0, nmda_ratio=3.0,
        exc_weight_scale=2.0, inh_weight_scale=2.5, depression_d=0.2, tau_d=500.0,
        noise_rate=5.0, noise_weight=0.007, adapt=True,
        gbar_kA_exc=0.006, gbar_kA_inh=0.004, tau_k=200.0,
        sahp_ainc_fast=0.005, sahp_tau_fast=300.0,
        sahp_ainc_slow=SAHP_NORMAL, sahp_tau_slow=6500.0, delay_per_distance=2.0,
    ),
    'sim': dict(dt=0.05, discard_transient_ms=1000.0),

    # ---- how much to generate ----------------------------------------------
    'states_to_run': ['normal', 'seizure'],   # trim to do one state at a time
    'n_recordings': 50,                       # per state; resumable
    'n_workers': 5,                           # concurrent NEURON processes
    'recording_ms': 60000.0,                  # kept length per recording

    # Per-neuron Poisson streams are keyed Random123(seed, gid, recording_index).
    # The recording index is the third key, so ONE seed already gives every
    # recording a different noise realisation. 1000 = the flagship's value, so
    # dephased recording NNN sees the same noise as flagship recording NNN.
    'noise_seed_base': 1000,

    # 'all' every cell ~77 MB/rec | 'probe' a subset ~3 MB/rec | 'none'
    'voltage': 'probe', 'voltage_probe_n': 40, 'voltage_dt': 5.0,

    # ---- warm start (this branch only) -------------------------------------
    # Run the network once for warmup_ms so it settles into its natural ongoing
    # state, saving every cell's state at snapshot_times. Recordings start from
    # those instead of finitialize(-65). Spacing is set against tau_slow = 6.5 s:
    # 50 s in is ~8 time constants (settled), 20 s apart is ~3 (so the snapshots
    # genuinely differ). EACH STATE NEEDS ITS OWN LIBRARY -- the seizure
    # network's stationary state is not the normal network's.
    'warmup_ms': 130000.0,
    'snapshot_times': [50000., 70000., 90000., 110000., 130000.],

    # Synaptic state is not restored, so each recording rebuilds recurrent
    # conductance from zero with un-depressed synapses. In ~1/3 of recordings
    # that ignites a burst at sim ~1 s -- just inside the kept window. Measured:
    # NOT snapshot-locked, so more snapshots would not help; the discard does.
    'discard_extra_ms': 3000.0,
}

CONFIG['states'] = {'normal': SAHP_NORMAL, 'seizure': SAHP_SEIZURE}
print('%s | %d rec x %.0f s per state | %d workers | voltage=%s'
      % (CONFIG['states_to_run'], CONFIG['n_recordings'],
         CONFIG['recording_ms'] / 1000, CONFIG['n_workers'], CONFIG['voltage']))


## 1. Preflight

Checks the interpreter, mechanisms, topology, libraries, and how much is left to
do. Nothing is generated until this says OK.

In [ ]:
pre = nb.preflight(CONFIG)


## 2. Build warm-start libraries

Once per state, ~2.5 h each, run concurrently. Skips any that already exist.

In [ ]:
nb.build_libraries(CONFIG, pre)
pre = nb.preflight(CONFIG)   # refresh


## 3. Generate

The long one. Resumable — rerun and it skips what already exists. Worker logs go
to `analysis/_dephase_nb_*.log`.

In [ ]:
nb.generate(CONFIG, pre)


## 4. Validate

Four gating checks per state: no Vm excursion at ~4.9 s, zero bursts in the
flagship's 4.60–5.34 s band, mean rate near 0.2789 Hz, V_rest near −83.3 mV.

In [ ]:
nb.validate(CONFIG)
